# Scoring by Player Origin - both previous team and hometown

## Dependencies and Setup

In [123]:
import os
import sys
from pathlib import Path
import pandas as pd
import regex as re

import numpy as np
import requests
from bs4 import BeautifulSoup
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
import matplotlib.font_manager as fm
from matplotlib.font_manager import FontProperties
from matplotlib.offsetbox import OffsetImage
from matplotlib.ticker import PercentFormatter
from matplotlib.ticker import ScalarFormatter
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from PIL import Image

# ======= BASE PATHS =======
try:
    # Works when running as a script
    base_dir = Path(__file__).resolve().parent
except NameError:
    # Fallback for notebooks or interactive mode
    base_dir = Path.cwd()

project_root = base_dir.parent.parent


# ======= DATA FOLDERS =======
temp_folder = project_root / "TEMP"
data_folder = project_root / "data"
roster_folder = data_folder / "player_info"
school_info_folder = data_folder / "school_info"

# ======= IMAGE FOLDERS =======
img_folder = project_root / "images"
logo_folder = img_folder / "logos"
background_folder = img_folder / "background"
plot_folder = project_root / "TEMP" / "IMAGE" / "scoring_origins"

# ======= IMPORT CONFIG =======
import config  # now you can import config.py

# ======= LOAD DATA =======
roster_file = roster_folder / "roster_10_30_25.csv"
roster_df = pd.read_csv(roster_file)
roster_df["Current Team"] = roster_df["Current Team"].replace("RPI", "Rensselaer")

print(roster_df.columns)

school_info_file = school_info_folder / "arena_school_info.csv"
school_info_df = pd.read_csv(school_info_file)

# Check the Config import
# print((config_folder / "config.py").read_text())

Index(['Current Team', 'Last_Name', 'First_Name', 'No', 'Position', 'Yr', 'Ht',
       'Wt', 'DOB', 'Hometown', 'Height_Inches', 'Draft_Year', 'NHL_Team',
       'D_Round', 'Last Team', 'League', 'City', 'State_Province', 'Country'],
      dtype='object')


## Transform the Roster data for easy Merge

In [124]:
# Examine the roster data
roster_df.head()
roster_df.info()
print(roster_df.columns)
# school_info_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1764 entries, 0 to 1763
Data columns (total 19 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Current Team    1764 non-null   object 
 1   Last_Name       1764 non-null   object 
 2   First_Name      1764 non-null   object 
 3   No              1764 non-null   int64  
 4   Position        1764 non-null   object 
 5   Yr              1764 non-null   object 
 6   Ht              1764 non-null   object 
 7   Wt              1764 non-null   int64  
 8   DOB             1764 non-null   object 
 9   Hometown        1764 non-null   object 
 10  Height_Inches   1764 non-null   int64  
 11  Draft_Year      247 non-null    float64
 12  NHL_Team        247 non-null    object 
 13  D_Round         247 non-null    float64
 14  Last Team       1764 non-null   object 
 15  League          1741 non-null   object 
 16  City            1764 non-null   object 
 17  State_Province  1764 non-null   o

In [125]:
## # Combine First and last name in roster_df to match player_ytd_df

# Clean white space from names
roster_df["First_Name"] = roster_df["First_Name"].str.strip()
roster_df["Last_Name"] = roster_df["Last_Name"].str.strip()
roster_df["Clean_Player"] = roster_df["First_Name"] + " " + roster_df["Last_Name"]
# Strip any leading/trailing whitespace
roster_df["Clean_Player"] = roster_df["Clean_Player"].str.strip()
# Rename Current Team to match player_ytd_df
roster_df = roster_df.rename(columns={"Current Team": "Team"})
# Reorder columns for easier viewing
#Order of columns
# ["No","Team","Clean_Player", "First_Name","Last_Name", 'Position', 'Yr', 'Ht', 'Wt', 'DOB', 'Hometown', 'Height_Inches', 'Draft_Year', 'NHL_Team', 'D_Round', 'Last Team', 'League', 'City', 'State_Province', 'Country']
roster_df = roster_df[["No","Team","Clean_Player", "First_Name","Last_Name", 'Position', 'Yr', 'Ht', 'Wt', 'DOB', 'Hometown', 'Height_Inches', 'Draft_Year', 'NHL_Team', 'D_Round', 'Last Team', 'League', 'City', 'State_Province', 'Country']]    


        


roster_df.head()

,No,Team,Clean_Player,First_Name,Last_Name,Position,Yr,Ht,Wt,DOB,Hometown,Height_Inches,Draft_Year,NHL_Team,D_Round,Last Team,League,City,State_Province,Country
0,3,Michigan State,Sean Barnhill,Sean,Barnhill,Defensemen,Fr,6-6,215,1/8/2007,"Scottsdale, Ariz.",78,NaN,NaN,NaN,Dubuque,USHL,Scottsdale,Arizona,USA
1,9,Michigan State,Matt Basgall,Matt,Basgall,Defensemen,Sr,5-9,190,8/16/2002,"Lake Forest, Ill.",69,NaN,NaN,NaN,Tri-City,USHL,Lake Forest,Illinois,USA
2,2,Michigan State,Patrick Geary,Patrick,Geary,Defensemen,Jr,6-1,185,2/18/2004,"Hamburg, N.Y.",73,2024.0,BUF,6.0,Waterloo,USHL,Hamburg,New York,USA
3,14,Michigan State,Matthew Lahey,Matthew,Lahey,Defensemen,Fr,6-6,205,7/17/2006,"Victoria, B.C.",78,2024.0,TOR,7.0,Fargo,USHL,Victoria,British Columbia,Canada
4,4,Michigan State,Colin Ralph,Colin,Ralph,Defensemen,So,6-4,210,10/4/2005,"Maple Grove, Minn.",76,2024.0,STL,2.0,St. Cloud State,NCHC,Maple Grove,Minnesota,USA


### Connect to database

In [126]:
## Connect to database using the recent_clean_db path from config.py
import sqlite3

#### CONFIG FILE NOTE WORKING AS EXPECTED - MANUAL FIX ####
data_folder = ('../../data/db/')
# filename = '2025_Feb_13_CLEAN.db'
filename = 'Nov_09_Current_Season_YTD_ROUGH.db'
recent_clean_db = data_folder + filename
########### END MANUAL FIX ###########

conn = sqlite3.connect(recent_clean_db)
cursor = conn.cursor()
print("Connected to database:", config.recent_clean_db)


Connected to database: ../../data/db/Nov_09_Current_Season_YTD_ROUGH.db


### Extract and merge the year to date stats
- Issue - the way the player stats ytd table is created it gives credit for games played to everyone, even if they didn't appear in a game
- ultimately I should change the scraping and aggrigation code so a player only gets credit for a game if TOI_sec is > 0

In [127]:
#### Extract and merge the year to date stats ####
player_ytd_query = """
SELECT * FROM player_stats_ytd
"""

player_ytd_df = pd.read_sql_query(player_ytd_query, conn)
# Replace RPI with Rensselaer to match roster_df
player_ytd_df["Team"] = player_ytd_df["Team"].replace("RPI", "Rensselaer")

## print length of DataFrame and columns
print("Player Stats YTD DataFrame shape:", player_ytd_df.shape)
print(player_ytd_df.columns)
# Drop any rows with TOTAL in the Clean_Player column
player_ytd_df = player_ytd_df[~player_ytd_df['Clean_Player'].str.contains('TOTAL', na=False)]

# Check length of DataFrame and columns after drop

print("Player Stats YTD DataFrame shape:", player_ytd_df.shape)
print(player_ytd_df.columns)

# Close the database connection
conn.close()



Player Stats YTD DataFrame shape: (1579, 14)
Index(['Clean_Player', 'Team', 'G', 'A', 'Pts', 'PlusMinus', 'Shots',
       'TOI_sec', 'PIM', 'FOW', 'FOL', 'Games_Played', 'FO%', 'TOI'],
      dtype='object')
Player Stats YTD DataFrame shape: (1579, 14)
Index(['Clean_Player', 'Team', 'G', 'A', 'Pts', 'PlusMinus', 'Shots',
       'TOI_sec', 'PIM', 'FOW', 'FOL', 'Games_Played', 'FO%', 'TOI'],
      dtype='object')


In [128]:
# Make sure name and team columns are stripped of punctuation, strange characters, and whitespace
player_ytd_df["Clean_Player"] = player_ytd_df["Clean_Player"].str.strip()
player_ytd_df["Team"] = player_ytd_df["Team"].str.strip()
roster_df["Clean_Player"] = roster_df["Clean_Player"].str.strip()
roster_df["Team"] = roster_df["Team"].str.strip()
# Remove any hyphens, periods, ect from team names to match
player_ytd_df["Team"] = player_ytd_df["Team"].str.replace(r'[^\w\s]', ' ', regex=True)
roster_df["Team"] = roster_df["Team"].str.replace(r'[^\w\s]', ' ', regex=True)
# QUICK FIX - Standardize team names with double spaces
# If team name column has double spaces, replace with single space
player_ytd_df["Team"] = player_ytd_df["Team"].str.replace('  ', ' ', regex=False)
roster_df["Team"] = roster_df["Team"].str.replace('  ', ' ', regex=False)

## Merge the two DataFrames on Clean_Player and Team
merged_df = pd.merge(
    player_ytd_df,
    roster_df,
    left_on=["Clean_Player", "Team"],
    right_on=["Clean_Player", "Team"],
    how="left"
)

# Shape and info of merged DataFrame
print("Merged DataFrame shape:", merged_df.shape)
# print(merged_df.columns)
# print(merged_df.head())
merged_df.info()



Merged DataFrame shape: (1579, 32)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1579 entries, 0 to 1578
Data columns (total 32 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Clean_Player    1579 non-null   object 
 1   Team            1579 non-null   object 
 2   G               1579 non-null   int64  
 3   A               1579 non-null   int64  
 4   Pts             1579 non-null   int64  
 5   PlusMinus       1579 non-null   int64  
 6   Shots           1579 non-null   int64  
 7   TOI_sec         1579 non-null   float64
 8   PIM             1579 non-null   int64  
 9   FOW             1579 non-null   float64
 10  FOL             1579 non-null   float64
 11  Games_Played    1579 non-null   int64  
 12  FO%             800 non-null    float64
 13  TOI             1579 non-null   object 
 14  No              1578 non-null   float64
 15  First_Name      1578 non-null   object 
 16  Last_Name       1578 non-null   object 
 17

### NEED TO DEAL WITH THIS WEIRD ONE EDGE CASE

In [129]:
## Export merged DataFrame to CSV for examination
# output_file = "../../TEMP/merged_player_stats_roster_test_1.csv"

## Show me a random selection of ten rows that didn't match for merge
## ie no current team, last name, ect
missing_team_df = merged_df[merged_df["First_Name"].isnull()]

missing_team_df


,Clean_Player,Team,G,A,Pts,PlusMinus,Shots,TOI_sec,PIM,FOW,...,Hometown,Height_Inches,Draft_Year,NHL_Team,D_Round,Last Team,League,City,State_Province,Country
1070,Maxon Vig,Bemidji State,1,1,2,8,11,11281.0,8,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Data Cleaning and Validation

### Filter out players with no TOI and goalies
- Remove players that haven't appeared in a game at all this year (TOI_sec = 0)
    -can't depend on YTD stats games played until we fix the scraping code
- Filter out goaltenders as non skaters - looking at offensive production so goalies are irrelivant (despite the assists they may get from time to time)



In [130]:
## Print Value COunt of Position column
print(merged_df['Position'].value_counts())

Position
Forwards       925
Defensemen     525
Goaltenders    128
Name: count, dtype: int64


In [131]:
### Print dataframe stats for to keep track of filtering steps
original_count = merged_df.shape[0]
print("Merged DataFrame shape before filtering:", merged_df.shape)

### Filter out players with no TOI and goalies
# Remove players that haven't appeared in a game at all this year (TOI_sec = 0)
# Remove any Rows where TOI_sec is 0 or NaN - These are goalies or players with no time on ice
merged_df = merged_df[(merged_df["TOI_sec"] > 0) & (~merged_df["TOI_sec"].isna())]
first_step_count = merged_df.shape[0]

## Check the shape after filtering
print("Merged DataFrame shape after filtering TOI_sec > 0:", merged_df.shape)
# Number of players removed
print("Players removed after filtering TOI_sec > 0:", original_count - first_step_count)

### DO NOT NEED TO FILTER FOR GOALIES BECAUSE PLAYER_YTD_STATS TABLE ONLY HAS TOI FOR SKATERS
# Filter out goaltenders as non skaters - looking at offensive production so goalies are irrelivant (despite the assists they may get from time to time)
# Strip any whitespace from Position column
# merged_df["Position"] = merged_df["Position"].str.strip()
# merged_df = merged_df[merged_df["Position"] != "Goaltenders"]
# no_goalie_count = merged_df.shape[0]
## Check the shape after filtering
# print("Merged DataFrame shape after filtering out Goalies:", merged_df.shape)
# # Number of players removed
# print("Players removed by filtering out Goalies:", first_step_count - no_goalie_count)

Merged DataFrame shape before filtering: (1579, 32)
Merged DataFrame shape after filtering TOI_sec > 0: (1446, 32)
Players removed after filtering TOI_sec > 0: 133


In [132]:
# merged_df.columns

## Quick Explore of Origin Data

In [133]:
## Value count of 'League' column
print(merged_df['League'].value_counts())

### Country value counts
print(merged_df['Country'].value_counts())

# # Check for any players that have null or 0 TOI but have Games Played > 0
# null_toi_df = merged_df[(merged_df["TOI_sec"].isnull()) | (merged_df["TOI_sec"] == 0) & (merged_df["Games_Played"] > 0)]
# null_toi_df




League
USHL                  434
BCHL                  227
NAHL                  184
WHL                    68
HEA                    56
OHL                    48
NCHC                   45
NTDP                   40
AJHL                   39
CCHA                   37
USports                34
ECAC                   33
QMJHL                  31
AHA                    27
B10                    21
OJHL                   15
D-I Ind.               13
SJHL                   12
NCDC                    8
MJHL                    8
Independents            8
CCHL                    6
J20 Nationell           5
SM-sarja                5
ECHL                    3
SHL                     2
AHL                     2
NCAA D3                 1
USHS                    1
AJHL/BCHL               1
CISAA                   1
JPL-Pro                 1
PREP                    1
D-III                   1
U20 SM-sarja            1
J18 Region              1
PHC                     1
DIII                    1
MHL  

### Clean and Classify Previous Team / League columns
- put into the same bins as used on the Team Construction Visual
- Copied code from there with a few changes made based on feedback / missed classifications 

In [134]:
### Reusing League and Team Classification function and libraries from team_construction_visual_workbook import classify_previous_team, classify_previous_league

# ----------------------------
# 1) Rename merged_df to df for easier to fit in with existing code
# -----------------------------
df = merged_df.copy()

# -----------------------------
# 2) Classification helpers
# -----------------------------
def _norm(s: str) -> str:
    if pd.isna(s):
        return ""
    s = str(s).strip().upper()
    s = re.sub(r"[.\u2010-\u2015\-–—]+", " ", s)  # unify hyphen-like chars to space
    s = re.sub(r"\s+", " ", s).strip()
    return s

NTDP_TEAM_HINTS = (
    "USA U 18", "US U 18", "USA U18", "US U18",
    "USA U 17", "US U 17", "USA U17", "US U17",
    "NTDP", "USNTDP", "US NATIONAL TEAM", "US DEV PROGRAM", "US DEVELOPMENT PROGRAM"
)
NTDP_LEAGUE_HINTS = ("NTDP",)

D1_CONFS = {"ECAC","CCHA","NCHC","HEA","B10","AHA","INDEPENDENTS","D I IND","NCAA"}

CJHL_LEAGUES = {"BCHL","AJHL","SJHL","OJHL", "MJHL", "CCHL","MHL"}

US_TIER1 = {"USHL"}                    # non-NTDP USHL here
US_TIER2 = {"NAHL","NCDC"}
US_OTHER  = {"USPHL","NA3HL"}    # common Tier III/independent

CHL = {"OHL","WHL","QMJHL", "MJAHL"}

EURO_HINTS = {
    "J20 NATIONELL","J18 REGION","U20 SM SARJA","U18","U20","SM SARJA",
    "SHL","ICEHL","ALPSHL","LIIGA","MHL RUSSIA","KHL, ICEHL","KHL","DEL", "EC-KAC"
}

RUSSIAN_MHL_TEAM_HINTS = (
    "KRASNAYA","LOKO","MOSKVA","MOSCOW","ST PETERSBURG","SKA","LOKOMOTIV",
    "DMITROV","CHELYABINSK","OMSK","NOVOSIBIRSK","MAGNITOGORSK","NIZHNY","YAROSLAVL",
)

BIN_ORDER = [
    "NTDP",
    "USHL (non‑NTDP)",
    "NAHL/NCDC",
    "US Juniors (Other)",
    "CHL (Major Junior)",
    "CJHL (Canadian Jr A)",
    "U SPORTS",
    "Europe",
    "NCAA D1 Transfers",
    "Other/Various/Unknown",
]

COLOR_MAP = {
    "NTDP": "#0057B8",
    "USHL (non‑NTDP)": "#1E90FF",
    "NAHL/NCDC": "#63B8FF",
    "US Juniors (Other)": "#B0E2FF",
    "CHL (Major Junior)": "#B22222",
    "CJHL (Canadian Jr A)": "#FF7F7F",
    "U SPORTS": "#FADBD8",
    "Europe": "#F0E130",
    "NCAA D1 Transfers": "#696969",
    "Other/Various/Unknown": "#000000",
}

def classify_prev_bin(last_team: str, league: str) -> str:
    t = _norm(last_team)
    l = _norm(league)

    # NTDP carve-out first
    if any(h in t for h in NTDP_TEAM_HINTS) or any(h == l for h in NTDP_LEAGUE_HINTS):
        return "NTDP"

    # NCAA D1 transfers
    if (l in D1_CONFS) or ("NCAA" in l and l != ""):
        return "NCAA D1 Transfers"

    # U SPORTS
    if l in {"USPORTS", "U SPORTS"}:
        return "U SPORTS"

    # CHL
    if l in CHL:
        return "CHL (Major Junior)"

    # MHL ambiguity
    if l == "MHL":
        if any(k in t for k in RUSSIAN_MHL_TEAM_HINTS):
            return "Europe"
        else:
            return "CJHL (Canadian Jr A)"

    # CJHL
    if l in CJHL_LEAGUES:
        return "CJHL (Canadian Jr A)"



    # US juniors
    if l in US_TIER1:
        return "USHL (non‑NTDP)"
    if l in US_TIER2:
        return "NAHL/NCDC"
    if (l in US_OTHER) or ("USPHL" in l) or ("NA3HL" in l) or (l == "EHL"):
        return "US Juniors (Other)"

    # Europe consolidated
    if (l in EURO_HINTS) or any(h in l for h in EURO_HINTS):
        return "Europe"


    # D3/Prep -> Other
    if l in {"D III","DIII","NCAA D3","PREP","USHS","CISAA","PHC"}:
        return "Other/Various/Unknown"

    # Fallbacks/Unknown
    if l == "" or pd.isna(league):
        return "Other/Various/Unknown"
    if any(h in t for h in ("PREP","HS","HIGH SCHOOL","CISAA","PHC")):
        return "Other/Various/Unknown"
    if "U SPORTS" in t:
        return "U SPORTS"
    if "IF Sundsvall" in t:  # edge case
        return "Europe"


    return "Other/Various/Unknown"


# Apply classification to the raw roster
df["Prev_League_Bin"] = df.apply(lambda r: classify_prev_bin(r.get("Last Team", np.nan),
                                                             r.get("League", np.nan)), axis=1)
df["Prev_League_Bin"] = pd.Categorical(df["Prev_League_Bin"], categories=BIN_ORDER, ordered=True)

# Save classified to temp_folder
# timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
# out_file = os.path.join(temp_folder, f"roster_classified_{timestamp}.csv")
# df.to_csv(out_file, index=False)


# show quick counts
overall_counts = (df["Prev_League_Bin"]
                  .value_counts(dropna=False)
                  .reindex(BIN_ORDER)
                  .fillna(0).astype(int))

print("Overall Counts by Prev_League_Bin")
print(overall_counts.reset_index().rename(columns={"index":"Bin","Prev_League_Bin":"Count"}))

Overall Counts by Prev_League_Bin
                   Count  count
0                   NTDP     41
1        USHL (non‑NTDP)    433
2              NAHL/NCDC    192
3     US Juniors (Other)      0
4     CHL (Major Junior)    147
5   CJHL (Canadian Jr A)    308
6               U SPORTS     34
7                 Europe     16
8      NCAA D1 Transfers    241
9  Other/Various/Unknown     34


In [135]:
## Show the US Juniors (Other) players to see what leagues they are from
us_juniors_df = df[df["Prev_League_Bin"] == "US Juniors (Other)"]
us_juniors_league_counts = (us_juniors_df["League"])
print("US Juniors (Other) League Counts")
print(us_juniors_league_counts.value_counts())

US Juniors (Other) League Counts
Series([], Name: count, dtype: int64)


In [136]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1446 entries, 0 to 1578
Data columns (total 33 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   Clean_Player     1446 non-null   object  
 1   Team             1446 non-null   object  
 2   G                1446 non-null   int64   
 3   A                1446 non-null   int64   
 4   Pts              1446 non-null   int64   
 5   PlusMinus        1446 non-null   int64   
 6   Shots            1446 non-null   int64   
 7   TOI_sec          1446 non-null   float64 
 8   PIM              1446 non-null   int64   
 9   FOW              1446 non-null   float64 
 10  FOL              1446 non-null   float64 
 11  Games_Played     1446 non-null   int64   
 12  FO%              800 non-null    float64 
 13  TOI              1446 non-null   object  
 14  No               1445 non-null   float64 
 15  First_Name       1445 non-null   object  
 16  Last_Name        1445 non-null   object  
 17  

### Adgrigate Stats using the new class bins

In [137]:
### Group by Prev_League_Bin and aggregate stats (G, A, Pts, PlusMinus, Shots, TOI_sec, PIM)

agg_stats = {
    "Games_Played": "sum",
    "G": "sum",
    "A": "sum",
    "Pts": "sum",
    "PlusMinus": "sum",
    "Shots": "sum",
    "TOI_sec": "sum",
    "PIM": "sum",
}
agg_df = df.groupby("Prev_League_Bin").agg(agg_stats).reset_index()
# # creat column for count of players in each bin
# agg_df["Player_Count"] = df["Prev_League_Bin"].value_counts().reindex(agg_df["Prev_League_Bin"]).values
# # Create AVG games played per player column
# agg_df["Avg_Games Played"] = agg_df["Games_Played"] / agg_df["Player_Count"]

# # Reorder Columns to put player_count right after Prev_League_Bin
# agg_df = agg_df[["Prev_League_Bin", "Player_Count", "Games_Played", "Avg_Games Played"] + list(agg_stats.keys())]


agg_df

C:\Users\jbanc\AppData\Local\Temp\ipykernel_20396\1377335929.py:13: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  agg_df = df.groupby("Prev_League_Bin").agg(agg_stats).reset_index()


,Prev_League_Bin,Games_Played,G,A,Pts,PlusMinus,Shots,TOI_sec,PIM
0,NTDP,376,96,120,216,26,714,371664.0,282
1,USHL (non‑NTDP),3386,537,947,1484,194,5467,3199004.0,1879
2,NAHL/NCDC,1236,126,242,368,-168,1585,1028001.0,664
3,US Juniors (Other),0,0,0,0,0,0,0.0,0
4,CHL (Major Junior),1225,209,359,568,-14,2115,1127930.0,792
5,CJHL (Canadian Jr A),2083,269,466,735,-141,2877,1779376.0,983
6,U SPORTS,275,51,80,131,-34,454,266383.0,208
7,Europe,109,19,27,46,3,144,92757.0,57
8,NCAA D1 Transfers,2079,350,573,923,-37,3504,2038378.0,1237
9,Other/Various/Unknown,269,48,78,126,29,475,262831.0,167


In [138]:
## Get count of players in each Prev_League_Bin
player_counts = (df["Prev_League_Bin"]
                  .value_counts(dropna=False)
                  .reindex(BIN_ORDER)
                  .fillna(0).astype(int))

# print("Player Counts by Prev_League_Bin")
# print(player_counts.reset_index().rename(columns={"index":"Bin","Prev_League_Bin":" Count"}))

In [139]:
#### Calulate per 60 minutes stats for each aggregated stat
for stat in agg_stats.keys():
    agg_df[stat + "_per_60min"] = (agg_df[stat] / agg_df["TOI_sec"]) * 3600
    

In [140]:
### Divide aggregated stats by player counts to get per player averages
for stat in agg_stats.keys():
    agg_df[stat + "_per_player"] = agg_df[stat] / player_counts.values
# agg_df

In [141]:
## Use the Games_Played column to get per game averages for all stats
for stat in agg_stats.keys():
    agg_df[stat + "_per_player_per_game_played"] = agg_df[stat] / agg_df["Games_Played"]




In [142]:
# creat column for count of players in each bin
agg_df["Player_Count"] = df["Prev_League_Bin"].value_counts().reindex(agg_df["Prev_League_Bin"]).values
# Create AVG games played per player column
agg_df["Avg_Games Played"] = agg_df["Games_Played"] / agg_df["Player_Count"]

# Move Player_Count column to be right after Prev_League_Bin and Avg_Games Played
cols = agg_df.columns.tolist()
cols.insert(1, cols.pop(cols.index("Player_Count")))
cols.insert(3, cols.pop(cols.index("Avg_Games Played")))
agg_df = agg_df[cols]

In [143]:
### Examine the final aggregated DataFrame
agg_df.info()
agg_df

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 35 columns):
 #   Column                                   Non-Null Count  Dtype   
---  ------                                   --------------  -----   
 0   Prev_League_Bin                          10 non-null     category
 1   Player_Count                             10 non-null     int64   
 2   Games_Played                             10 non-null     int64   
 3   Avg_Games Played                         9 non-null      float64 
 4   G                                        10 non-null     int64   
 5   A                                        10 non-null     int64   
 6   Pts                                      10 non-null     int64   
 7   PlusMinus                                10 non-null     int64   
 8   Shots                                    10 non-null     int64   
 9   TOI_sec                                  10 non-null     float64 
 10  PIM                                      

,Prev_League_Bin,Player_Count,Games_Played,Avg_Games Played,G,A,Pts,PlusMinus,Shots,TOI_sec,...,TOI_sec_per_player,PIM_per_player,Games_Played_per_player_per_game_played,G_per_player_per_game_played,A_per_player_per_game_played,Pts_per_player_per_game_played,PlusMinus_per_player_per_game_played,Shots_per_player_per_game_played,TOI_sec_per_player_per_game_played,PIM_per_player_per_game_played
0,NTDP,41,376,9.170732,96,120,216,26,714,371664.0,...,9064.975610,6.878049,1.0,0.255319,0.319149,0.574468,0.069149,1.898936,988.468085,0.750000
1,USHL (non‑NTDP),433,3386,7.819861,537,947,1484,194,5467,3199004.0,...,7388.000000,4.339492,1.0,0.158594,0.279681,0.438275,0.057295,1.614589,944.773774,0.554932
2,NAHL/NCDC,192,1236,6.437500,126,242,368,-168,1585,1028001.0,...,5354.171875,3.458333,1.0,0.101942,0.195793,0.297735,-0.135922,1.282362,831.716019,0.537217
3,US Juniors (Other),0,0,NaN,0,0,0,0,0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,CHL (Major Junior),147,1225,8.333333,209,359,568,-14,2115,1127930.0,...,7672.993197,5.387755,1.0,0.170612,0.293061,0.463673,-0.011429,1.726531,920.759184,0.646531
5,CJHL (Canadian Jr A),308,2083,6.762987,269,466,735,-141,2877,1779376.0,...,5777.194805,3.191558,1.0,0.129141,0.223716,0.352856,-0.067691,1.381181,854.237158,0.471916
6,U SPORTS,34,275,8.088235,51,80,131,-34,454,266383.0,...,7834.794118,6.117647,1.0,0.185455,0.290909,0.476364,-0.123636,1.650909,968.665455,0.756364
7,Europe,16,109,6.812500,19,27,46,3,144,92757.0,...,5797.312500,3.562500,1.0,0.174312,0.247706,0.422018,0.027523,1.321101,850.981651,0.522936
8,NCAA D1 Transfers,241,2079,8.626556,350,573,923,-37,3504,2038378.0,...,8458.000000,5.132780,1.0,0.168350,0.275613,0.443963,-0.017797,1.685426,980.460798,0.594998
9,Other/Various/Unknown,34,269,7.911765,48,78,126,29,475,262831.0,...,7730.323529,4.911765,1.0,0.178439,0.289963,0.468401,0.107807,1.765799,977.066914,0.620818


### Plan for Visualizations
- data transformation is looking good to this point. SHould have a few diff. interesting views

- Should I use stacked bars witht he country bins and color grtadiant within?
- Should come up a good visual for the number of games played by each bin of players

## Bin By Country

In [144]:
### Do The Aggrigation by Home Country
country_agg_stats = {
    "Games_Played": "sum",
    "G": "sum",
    "A": "sum",
    "Pts": "sum",
    "PlusMinus": "sum",
    "Shots": "sum",
    "TOI_sec": "sum",
    "PIM": "sum",
}
country_agg_df = merged_df.groupby("Country").agg(country_agg_stats).reset_index()
print(country_agg_df.head(20))

## Get count of players in each Country and calculate per player averages
country_player_counts = (df["Country"]
                    .value_counts(dropna=False)
                    .fillna(0).astype(int))
# print("Player Counts by Country")
# print(country_player_counts.reset_index().rename(columns={"index":"Country","Country":" Count"}))

# ### Divide aggregated stats by player counts to get per player averages by Country
# country_player_counts = country_player_counts.reindex(country_agg_df.index)
# for stat in country_agg_stats.keys():
#     country_agg_df[stat + "_per_player"] = country_agg_df[stat] / country_player_counts

    


           Country  Games_Played    G     A   Pts  PlusMinus  Shots  \
0          Austria            23    7    10    17          8     44   
1          Belarus            15    4     4     8         -1     15   
2              CYM            11    0     4     4         -5     10   
3           Canada          4082  647  1119  1766       -133   6549   
4          Croatia             7    0     1     1          3     15   
5   Czech Republic            54   13    16    29         -3    122   
6          Czechia            11    1     2     3         -8     21   
7          Finland           140   24    36    60         10    199   
8          Germany             9    0     2     2         -2     12   
9    Great Britain             7    0     0     0          1      9   
10         Hungary             8    1     2     3          4      7   
11           Italy            10    0     0     0         -7     14   
12             JPN             7    0     1     1         -1      6   
13    

In [145]:
## Calculate rate stats per 60 minutes for each aggregated stat by Country
for stat in country_agg_stats.keys():
    country_agg_df[stat + "_per_60min"] = (country_agg_df[stat] / country_agg_df["TOI_sec"]) * 3600

## Calculate per player averages for each aggregated stat by Country
country_player_counts = country_player_counts.reindex(country_agg_df["Country"])
for stat in country_agg_stats.keys():
    country_agg_df[stat + "_per_player"] = country_agg_df[stat] / country_player_counts.values

# Calculate per game averages for each aggregated stat by Country
for stat in country_agg_stats.keys():
    country_agg_df[stat + "_per_player_per_game_played"] = country_agg_df[stat] / country_agg_df["Games_Played"]

country_agg_df.info()
country_agg_df.head(20)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 33 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   Country                                  24 non-null     object 
 1   Games_Played                             24 non-null     int64  
 2   G                                        24 non-null     int64  
 3   A                                        24 non-null     int64  
 4   Pts                                      24 non-null     int64  
 5   PlusMinus                                24 non-null     int64  
 6   Shots                                    24 non-null     int64  
 7   TOI_sec                                  24 non-null     float64
 8   PIM                                      24 non-null     int64  
 9   Games_Played_per_60min                   24 non-null     float64
 10  G_per_60min                              24 non-null

,Country,Games_Played,G,A,Pts,PlusMinus,Shots,TOI_sec,PIM,Games_Played_per_60min,...,TOI_sec_per_player,PIM_per_player,Games_Played_per_player_per_game_played,G_per_player_per_game_played,A_per_player_per_game_played,Pts_per_player_per_game_played,PlusMinus_per_player_per_game_played,Shots_per_player_per_game_played,TOI_sec_per_player_per_game_played,PIM_per_player_per_game_played
0,Austria,23,7,10,17,8,44,22769.0,4,3.636523,...,7589.666667,1.333333,1.0,0.304348,0.434783,0.739130,0.347826,1.913043,989.956522,0.173913
1,Belarus,15,4,4,8,-1,15,10860.0,4,4.972376,...,5430.000000,2.000000,1.0,0.266667,0.266667,0.533333,-0.066667,1.000000,724.000000,0.266667
2,CYM,11,0,4,4,-5,10,10907.0,4,3.630696,...,5453.500000,2.000000,1.0,0.000000,0.363636,0.363636,-0.454545,0.909091,991.545455,0.363636
3,Canada,4082,647,1119,1766,-133,6549,3743697.0,2423,3.925318,...,7050.276836,4.563089,1.0,0.158501,0.274130,0.432631,-0.032582,1.604361,917.123224,0.593582
4,Croatia,7,0,1,1,3,15,4396.0,2,5.732484,...,4396.000000,2.000000,1.0,0.000000,0.142857,0.142857,0.428571,2.142857,628.000000,0.285714
5,Czech Republic,54,13,16,29,-3,122,57773.0,30,3.364894,...,9628.833333,5.000000,1.0,0.240741,0.296296,0.537037,-0.055556,2.259259,1069.870370,0.555556
6,Czechia,11,1,2,3,-8,21,10394.0,2,3.809890,...,10394.000000,2.000000,1.0,0.090909,0.181818,0.272727,-0.727273,1.909091,944.909091,0.181818
7,Finland,140,24,36,60,10,199,122468.0,69,4.115361,...,5831.809524,3.285714,1.0,0.171429,0.257143,0.428571,0.071429,1.421429,874.771429,0.492857
8,Germany,9,0,2,2,-2,12,8761.0,2,3.698208,...,8761.000000,2.000000,1.0,0.000000,0.222222,0.222222,-0.222222,1.333333,973.444444,0.222222
9,Great Britain,7,0,0,0,1,9,4981.0,6,5.059225,...,4981.000000,6.000000,1.0,0.000000,0.000000,0.000000,0.142857,1.285714,711.571429,0.857143


## Bin By State / Province

In [146]:
## Examine the State_Province column value counts
state_prov_counts = merged_df['State_Province'].value_counts()
print("State/Province Value Counts")
print(state_prov_counts)

State/Province Value Counts
State_Province
Minnesota           181
Ontario             174
British Columbia    106
Alberta              98
New York             84
                   ... 
United Kingdom        1
Delaware              1
Czechia               1
Okla.                 1
RUS                   1
Name: count, Length: 69, dtype: int64


In [147]:

## Filter out any rows that don't have USA or CAN in the Country column

# print length of merged_df before filtering
print("Merged DataFrame shape before filtering for USA/CAN:", merged_df.shape)
us_can_df = merged_df[(merged_df["Country"] == "USA") | (merged_df["Country"] == "Canada")]

# print length of us_can_df after filtering
print("Filtered DataFrame shape for USA/CAN:", us_can_df.shape)

# Print summary of State_Province values in us_can_df
print("USA/CAN DataFrame State/Province Value Counts")
# USA / Canada Country Value Counts
print(us_can_df['Country'].value_counts())
# State Province Value Counts
print(us_can_df['State_Province'].value_counts())

# Check State Province values of rows filtered out
# filtered_out_df = merged_df[~merged_df.index.isin(us_can_df.index)]
# print("Filtered Out DataFrame State/Province Value Counts")
# print(filtered_out_df['State_Province'].value_counts())


Merged DataFrame shape before filtering for USA/CAN: (1446, 32)
Filtered DataFrame shape for USA/CAN: (1330, 32)
USA/CAN DataFrame State/Province Value Counts
Country
USA       799
Canada    531
Name: count, dtype: int64
State_Province
Minnesota                    181
Ontario                      174
British Columbia             106
Alberta                       98
New York                      84
Massachusetts                 69
Michigan                      68
Quebec                        66
Illinois                      53
California                    45
New Jersey                    37
Saskatchewan                  35
Pennsylvania                  29
Wisconsin                     26
Manitoba                      26
Connecticut                   24
New Hampshire                 14
Colorado                      14
Missouri                      13
North Dakota                  13
Ohio                          13
Maryland                      12
Alaska                        12
Nova 

In [148]:
### Group and aggregate stats by State_Province for USA and CAN players
state_prov_agg_stats = {
    "Games_Played": "sum",
    "G": "sum",
    "A": "sum",
    "Pts": "sum",
    "PlusMinus": "sum",
    "Shots": "sum",
    "TOI_sec": "sum",
    "PIM": "sum",
}

state_prov_agg_df = us_can_df.groupby("State_Province").agg(state_prov_agg_stats).reset_index()
print(state_prov_agg_df.head(20))

## Get count of players in each State_Province and calculate per player averages
state_prov_player_counts = (us_can_df["State_Province"]
                    .value_counts(dropna=False)
                    .fillna(0).astype(int))
print("Player Counts by State/Province")
print(state_prov_player_counts.reset_index().rename(columns={"index":"State/Province","State_Province":" Count"}))

### Divide aggregated stats by player counts to get per player averages by State_Province
state_prov_player_counts = state_prov_player_counts.reindex(state_prov_agg_df["State_Province"])
for stat in state_prov_agg_stats.keys():
    state_prov_agg_df[stat + "_per_player"] = state_prov_agg_df[stat] / state_prov_player_counts.values

## Calculate rate stats per 60 minutes for each aggregated stat by State_Province
for stat in state_prov_agg_stats.keys():
    state_prov_agg_df[stat + "_per_60min"] = (state_prov_agg_df[stat] / state_prov_agg_df["TOI_sec"]) * 3600

# Calculate per game averages for each aggregated stat by State_Province
for stat in state_prov_agg_stats.keys():
    state_prov_agg_df[stat + "_per_player_per_game_played"] = state_prov_agg_df[stat] / state_prov_agg_df["Games_Played"]
state_prov_agg_df.info()
state_prov_agg_df.head(20)

      State_Province  Games_Played    G    A  Pts  PlusMinus  Shots  \
0             Alaska            84   19   28   47          0    160   
1            Alberta           770  121  190  311        -51   1134   
2            Arizona            86    8   31   39          3     94   
3   British Columbia           796  113  206  319        -21   1237   
4         California           293   42   73  115         -2    424   
5           Colorado            88    6   14   20          7     94   
6        Connecticut           166   16   42   58          5    264   
7           Delaware             8    1    2    3          2      4   
8            Florida            88   19   37   56          8    152   
9            Georgia            47    5   12   17          1     65   
10             Idaho             9    1    2    3         -1      7   
11          Illinois           457   60  122  182         28    697   
12           Indiana            90   10   14   24        -16    106   
13    

,State_Province,Games_Played,G,A,Pts,PlusMinus,Shots,TOI_sec,PIM,Games_Played_per_player,...,TOI_sec_per_60min,PIM_per_60min,Games_Played_per_player_per_game_played,G_per_player_per_game_played,A_per_player_per_game_played,Pts_per_player_per_game_played,PlusMinus_per_player_per_game_played,Shots_per_player_per_game_played,TOI_sec_per_player_per_game_played,PIM_per_player_per_game_played
0,Alaska,84,19,28,47,0,160,86195.0,28,7.000000,...,3600.0,1.169441,1.0,0.226190,0.333333,0.559524,0.000000,1.904762,1026.130952,0.333333
1,Alberta,770,121,190,311,-51,1134,686523.0,424,7.857143,...,3600.0,2.223378,1.0,0.157143,0.246753,0.403896,-0.066234,1.472727,891.588312,0.550649
2,Arizona,86,8,31,39,3,94,75748.0,65,9.555556,...,3600.0,3.089190,1.0,0.093023,0.360465,0.453488,0.034884,1.093023,880.790698,0.755814
3,British Columbia,796,113,206,319,-21,1237,716463.0,439,7.509434,...,3600.0,2.205836,1.0,0.141960,0.258794,0.400754,-0.026382,1.554020,900.079146,0.551508
4,California,293,42,73,115,-2,424,246659.0,174,6.511111,...,3600.0,2.539538,1.0,0.143345,0.249147,0.392491,-0.006826,1.447099,841.839590,0.593857
5,Colorado,88,6,14,20,7,94,73582.0,56,6.285714,...,3600.0,2.739800,1.0,0.068182,0.159091,0.227273,0.079545,1.068182,836.159091,0.636364
6,Connecticut,166,16,42,58,5,264,148675.0,58,6.916667,...,3600.0,1.404406,1.0,0.096386,0.253012,0.349398,0.030120,1.590361,895.632530,0.349398
7,Delaware,8,1,2,3,2,4,6183.0,2,8.000000,...,3600.0,1.164483,1.0,0.125000,0.250000,0.375000,0.250000,0.500000,772.875000,0.250000
8,Florida,88,19,37,56,8,152,84875.0,72,8.800000,...,3600.0,3.053903,1.0,0.215909,0.420455,0.636364,0.090909,1.727273,964.488636,0.818182
9,Georgia,47,5,12,17,1,65,34533.0,55,6.714286,...,3600.0,5.733646,1.0,0.106383,0.255319,0.361702,0.021277,1.382979,734.744681,1.170213
